# Demo 3: Comparing Different Forecasting Approaches

In this demo, we'll compare different time series forecasting methods:
1. ARIMA models
2. Prophet
3. LSTM neural networks
4. Ensemble approach

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

# Set random seed
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn')
sns.set_palette('husl')

## 1. Generate Complex Time Series Data

Create synthetic health data with multiple seasonal patterns and trends.

In [ ]:
# Create date range for three months of hourly data
dates = pd.date_range(start='2024-01-01', end='2024-03-31', freq='H')

# Create complex patterns
hour_pattern = 10 * np.sin(2 * np.pi * (dates.hour - 6) / 24)
day_pattern = 5 * np.sin(2 * np.pi * dates.dayofweek / 7)
month_pattern = 3 * np.sin(2 * np.pi * dates.day / 30)
trend = np.linspace(0, 10, len(dates))

# Generate data with multiple patterns
data = pd.DataFrame({
    'ds': dates,  # Prophet requires 'ds' column name
    'y': 70 + hour_pattern + day_pattern + month_pattern + trend + np.random.normal(0, 2, len(dates))
})

# Plot the data
plt.figure(figsize=(15, 5))
plt.plot(data['ds'], data['y'])
plt.title('Synthetic Health Time Series Data')
plt.xlabel('Date')
plt.ylabel('Value')
plt.show()

## 2. ARIMA Model

Implement and evaluate an ARIMA model.

In [ ]:
def fit_arima(data, train_size=0.8):
    """Fit ARIMA model and make predictions."""
    # Split data
    train_size = int(len(data) * train_size)
    train = data[:train_size]
    test = data[train_size:]
    
    # Fit ARIMA model
    model = ARIMA(train['y'], order=(1, 1, 1))
    results = model.fit()
    
    # Make predictions
    predictions = results.forecast(steps=len(test))
    
    return train, test, predictions

# Fit ARIMA model
train_arima, test_arima, pred_arima = fit_arima(data)

# Plot results
plt.figure(figsize=(15, 5))
plt.plot(train_arima['ds'], train_arima['y'], label='Training Data')
plt.plot(test_arima['ds'], test_arima['y'], label='Actual')
plt.plot(test_arima['ds'], pred_arima, label='ARIMA Predictions')
plt.title('ARIMA Model Predictions')
plt.legend()
plt.show()

## 3. Prophet Model

Implement and evaluate a Prophet model.

In [ ]:
def fit_prophet(data, train_size=0.8):
    """Fit Prophet model and make predictions."""
    # Split data
    train_size = int(len(data) * train_size)
    train = data[:train_size]
    test = data[train_size:]
    
    # Fit Prophet model
    model = Prophet(daily_seasonality=True, weekly_seasonality=True)
    model.fit(train)
    
    # Make predictions
    future = model.make_future_dataframe(periods=len(test), freq='H')
    forecast = model.predict(future)
    
    return train, test, forecast

# Fit Prophet model
train_prophet, test_prophet, forecast_prophet = fit_prophet(data)

# Plot results
plt.figure(figsize=(15, 5))
plt.plot(train_prophet['ds'], train_prophet['y'], label='Training Data')
plt.plot(test_prophet['ds'], test_prophet['y'], label='Actual')
plt.plot(forecast_prophet['ds'], forecast_prophet['yhat'], label='Prophet Predictions')
plt.fill_between(forecast_prophet['ds'], 
                 forecast_prophet['yhat_lower'], 
                 forecast_prophet['yhat_upper'], 
                 alpha=0.2)
plt.title('Prophet Model Predictions with Confidence Intervals')
plt.legend()
plt.show()

# Plot Prophet components
model_prophet = Prophet(daily_seasonality=True, weekly_seasonality=True)
model_prophet.fit(data)
model_prophet.plot_components(forecast_prophet)

## 4. LSTM Model

Implement and evaluate an LSTM neural network model.

In [ ]:
def prepare_lstm_data(data, lookback=24):
    """Prepare data for LSTM model."""
    # Scale data
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data['y'].values.reshape(-1, 1))
    
    # Create sequences
    X, y = [], []
    for i in range(len(scaled_data) - lookback):
        X.append(scaled_data[i:(i + lookback), 0])
        y.append(scaled_data[i + lookback, 0])
    
    return np.array(X), np.array(y), scaler

def fit_lstm(data, train_size=0.8, lookback=24):
    """Fit LSTM model and make predictions."""
    # Prepare data
    X, y, scaler = prepare_lstm_data(data, lookback)
    
    # Split data
    train_size = int(len(X) * train_size)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    # Reshape data for LSTM [samples, time steps, features]
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
    
    # Build LSTM model
    model = Sequential([
        LSTM(50, activation='relu', input_shape=(lookback, 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    
    # Train model
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
    
    # Make predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    # Inverse transform predictions
    train_pred = scaler.inverse_transform(train_pred)
    test_pred = scaler.inverse_transform(test_pred)
    
    return train_pred, test_pred

# Fit LSTM model
train_pred_lstm, test_pred_lstm = fit_lstm(data)

# Plot results
plt.figure(figsize=(15, 5))
plt.plot(data['ds'][:len(train_pred_lstm)], data['y'][:len(train_pred_lstm)], label='Training Data')
plt.plot(data['ds'][len(train_pred_lstm):len(train_pred_lstm)+len(test_pred_lstm)], 
         data['y'][len(train_pred_lstm):len(train_pred_lstm)+len(test_pred_lstm)], 
         label='Actual')
plt.plot(data['ds'][len(train_pred_lstm):len(train_pred_lstm)+len(test_pred_lstm)], 
         test_pred_lstm, 
         label='LSTM Predictions')
plt.title('LSTM Model Predictions')
plt.legend()
plt.show()

## 5. Model Comparison

Compare the performance of all models.

In [ ]:
def calculate_metrics(actual, predicted):
    """Calculate performance metrics."""
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'MAE': mean_absolute_error(actual, predicted)
    }

# Calculate metrics for each model
metrics = {
    'ARIMA': calculate_metrics(test_arima['y'], pred_arima),
    'Prophet': calculate_metrics(test_prophet['y'], 
                                forecast_prophet['yhat'][-len(test_prophet):]),
    'LSTM': calculate_metrics(data['y'][len(train_pred_lstm):len(train_pred_lstm)+len(test_pred_lstm)],
                             test_pred_lstm.flatten())
}

# Create comparison DataFrame
metrics_df = pd.DataFrame(metrics).T

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
metrics_df['RMSE'].plot(kind='bar', ax=axes[0], title='RMSE Comparison')
metrics_df['MAE'].plot(kind='bar', ax=axes[1], title='MAE Comparison')
plt.tight_layout()
plt.show()

print("\nModel Performance Metrics:")
print(metrics_df)